In [2]:
import os

# DATASET
train_path = '/content/drive/MyDrive/computer vision/horse-or-human/train' 
test_path = '/content/drive/MyDrive/computer vision/horse-or-human/validation'
labels = os.listdir(train_path)

os.mkdir('checkpoints')

# MODEL
backbone = 'resnet18'
# num_classes = 2
num_classes = len(labels)

# TRAIN
epochs = 15
batch_size = 4
num_workers = 2
sch_patience = 5000
# lr = 0.00005
lr = 0.001

min_lr = 0.00005
factor = 0.97
# factor = 0.1

In [3]:
import os
from os.path import join as pj
from torchvision import transforms
from PIL import Image, ImageDraw
import torch
from torch.utils.data import Dataset as dts
import numpy as np
from matplotlib import pyplot as plt
import cv2
from random import randint, shuffle



class Dataset(dts):
    def __init__(self, data_dir, classes, is_shuffle= False, augs = False):
        self.data_dir = data_dir
        self.augs = augs
        self.label2idx = {label:idx for idx, label in enumerate(classes)}

        if self.augs:
            trns = [transforms.Resize((224, 224)),
                                       transforms.ColorJitter(hue=(-0.5,0.5)),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.RandomVerticalFlip(),
                                       transforms.ToTensor(),
                                       transforms.Lambda(lambda x: x[np.random.permutation(3), :, :]),
                                       transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]

        else:
            trns = [transforms.Resize((224, 224)),
                                     transforms.ToTensor(),
                                     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]

        self.preprocess = transforms.Compose(trns)
        
        self.whole_data = {}
        for class_dir in os.listdir(data_dir):
            for file_name in os.listdir(pj(data_dir, class_dir)):
                self.whole_data[pj(class_dir, file_name)] = self.label2idx[class_dir]
                # self.images.append(pj(class_dir, file_name))
                # self.labels.append(self.label2idx[class_dir])
        self.images = list(self.whole_data.keys())
        if is_shuffle:
            shuffle(self.images)
        self.labels = [self.whole_data[img] for img in self.images]
        

    def __getitem__(self, idx):
        p = pj(self.data_dir, self.images[idx])
        img = Image.open(p).convert('RGB')

        
        # plt.imshow(img)
        # plt.savefig('fig1.png')

        tensor = self.preprocess(img)
        label = torch.tensor(self.labels[idx])    
        return tensor, label


    def __len__(self,):
        return len(self.images)




In [4]:
from __future__ import print_function, division

import torch.nn as nn
import torch
import torchvision
import torch.nn.functional as F
#import torch.utils.model_zoo as model_zoo

class MyResNet(nn.Module):

    def __init__(self, num_classes):
        super(MyResNet, self).__init__()
        self.model = torchvision.models.resnet18()

        self.model.fc = nn.Sequential(nn.Linear(self.model.fc.in_features, 128),
                            nn.ReLU(),
                            nn.Dropout(0.2),
                            nn.Linear(128, num_classes),
                            )

    def forward(self, x):
        x = self.model(x)
        # x = F.softmax(x)

        return x


In [5]:
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
import os
import matplotlib.pyplot as plt
from os.path import join as pj
import numpy as np
from sklearn.utils import class_weight
from tqdm import tqdm



def mean(x):
    return sum(x)/len(x)

def round_dict(d, r):
    return {k:round(d[k], r) for k in d}

def count_class_weights(train_path):
    targets = []
    lbl2idx = {label:idx for idx, label in enumerate(os.listdir(train_path))}
    for label in os.listdir(train_path):
        for file_name in os.listdir(pj(train_path, label)):
            targets.append(lbl2idx[label])

    targets = np.array(targets)
    print(np.unique(targets, return_counts=True))

    class_weights = class_weight.compute_class_weight('balanced',
                                                 classes=np.unique(targets),
                                                 y=targets)
    return class_weights

def test(model, criterion, dataloader):
    losses = []
    predictions = []
    targets = []
    for i, batch in enumerate(tqdm(dataloader)):
        images, labels = (t.cuda() for t in batch)
        preds = model(images)

        loss = criterion(preds, labels)
        losses.append(loss.item())

        predictions.extend(preds.argmax(dim=1).tolist())
        targets.extend(labels.tolist())
    
    return mean(losses), targets, predictions


def train(model, optimizer, scheduler, criterion, dataloader):
    losses = []
    for i, batch in enumerate(tqdm(dataloader)):
        images, labels = (t.cuda() for t in batch)

        

        preds = model(images)

        optimizer.zero_grad()
        loss = criterion(preds, labels)

        losses.append(loss.item())

        loss.backward()
        optimizer.step()
        scheduler.step(loss.item())

    return model, mean(losses)

def count_metrics_by_classes(targets, predictions, num_classes, classes):
    accs_by_classes = {}
    label2idx = {label:idx for idx, label in enumerate(classes)}
    for label in classes:
        shots = []
        for t, p in zip(targets, predictions):
            if t == label2idx[label]:
                shots.append(t==p)

        accs_by_classes[label] = mean(shots)
    return accs_by_classes

def print_metrics(test_loss, targets, predictions, num_classes, labels, epoch, r=3):
    
    accuracy = accuracy_score(targets, predictions)
    ballanced_accuracy = balanced_accuracy_score(targets, predictions)

   

    print('TEST LOSS:', test_loss)
    print('ACCURACY:', round(accuracy,r))
    print('BALLANCED ACCURACY:', round(ballanced_accuracy,r))


  

    classes_metrics = count_metrics_by_classes(targets, predictions, num_classes, labels)

    print('**************************METRICS BY CLASSES************************')
    print('ACCURACY:', round_dict(classes_metrics, r))




In [6]:
import torch
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from os.path import join as pj
from pprint import pprint
import os
import numpy as np

        
def main():
    print(labels)

    train_dataset = Dataset(train_path, labels, is_shuffle=True, augs=True)
    test_dataset = Dataset(test_path, labels)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, num_workers=num_workers)


    class_weights = count_class_weights(train_path)
    criterion = nn.CrossEntropyLoss(weight = torch.tensor(class_weights).float(),reduction='mean')
    criterion = criterion.cuda()
    
    model = MyResNet(num_classes)
    model = model.cuda()

    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=factor, \
         patience=sch_patience, verbose=True, min_lr=min_lr)

    best_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        model, train_loss = train(model, optimizer, scheduler, criterion, train_dataloader)

        model.eval()
        test_loss, targets, predictions = test(model, criterion, test_dataloader)

        print_metrics(test_loss, targets, predictions, num_classes, labels, epoch)

        if test_loss < best_loss:
            best_loss = test_loss
            torch.save(model, f'checkpoints/model_{epoch}.pth')

In [7]:
main()

['horses', 'humans']
(array([0, 1]), array([220, 527]))


100%|██████████| 64/64 [00:30<00:00,  2.09it/s]


TEST LOSS: 1.2213317632395047
ACCURACY: 0.531
BALLANCED ACCURACY: 0.531
**************************METRICS BY CLASSES************************
ACCURACY: {'horses': 0.062, 'humans': 1.0}


100%|██████████| 64/64 [00:02<00:00, 29.15it/s]


TEST LOSS: 0.4230677755258512
ACCURACY: 0.848
BALLANCED ACCURACY: 0.848
**************************METRICS BY CLASSES************************
ACCURACY: {'horses': 0.703, 'humans': 0.992}


...

TEST LOSS: 0.3159900927939816
ACCURACY: 0.91
BALLANCED ACCURACY: 0.91
**************************METRICS BY CLASSES************************
ACCURACY: {'horses': 0.82, 'humans': 1.0}


100%|██████████| 64/64 [00:02<00:00, 28.10it/s]


TEST LOSS: 0.31711401727079647
ACCURACY: 0.902
BALLANCED ACCURACY: 0.902
**************************METRICS BY CLASSES************************
ACCURACY: {'horses': 0.805, 'humans': 1.0}


100%|██████████| 64/64 [00:02<00:00, 27.58it/s]

TEST LOSS: 0.5066996744194512
ACCURACY: 0.898
BALLANCED ACCURACY: 0.898
**************************METRICS BY CLASSES************************
ACCURACY: {'horses': 0.797, 'humans': 1.0}
